# 01 · Ingesta

Descarga las cuatro fuentes del proyecto a `data/raw/` y las lee con PySpark para verificar esquema, volumen y cobertura temporal.

| Fuente | Institución | Formato | Periodicidad |
| :--- | :--- | :--- | :--- |
| Ingresos y gastos del Gobierno Federal | SHCP | CSV | Mensual |
| Ingresos tributarios | SAT | XLS → CSV | Mensual |
| IGAE (base 2018) | INEGI | CSV (zip) | Mensual |
| INPC | INEGI | CSV (zip) | Mensual |

Ejecutar desde `notebooks/`. Los datos no se versionan (`data/` está en `.gitignore`); este notebook es la forma de obtenerlos. Detalle de cada fuente en [docs/fuentes.md](../docs/fuentes.md).

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

import shutil
import ssl
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

PERIODO = (201501, 202503)  # AAAAMM; termina en 2025-03 porque el IGAE de datos abiertos no tiene meses posteriores

print("Datos crudos en:", RAW)

In [ ]:
# ============================================================
# 2. FUENTES
# ============================================================

FUENTES = {
    "ingreso_gasto_finan.csv": "https://www.secciones.hacienda.gob.mx/work/models/estadisticas_oportunas/datos_abiertos_eopf/ingreso_gasto_finan.csv",
    "IngresosTributarios.xls": "https://wu1agsprosta001.blob.core.windows.net/agsc-publicaciones/Datos_abiertos/Cifras_SAT/Documents/IngresosTributarios.xls",
    "igae_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/igae/2018/datosabiertos/igae_mensual_csv.zip",
    "inpc_mensual_csv.zip": "https://www.inegi.org.mx/contenidos/programas/inpc/2018a/datosabiertos/conjunto_de_datos_inpc_indicador_mensual_csv.zip",
}

In [ ]:
# ============================================================
# 3. DESCARGA
# ============================================================
#
# El servidor de la SHCP no envía el certificado intermedio
# correcto (Let's Encrypt YR1). En lugar de desactivar la
# verificación SSL, descargamos ese intermedio y lo agregamos
# al contexto: la conexión sigue verificada contra la raíz.
#
# Si el archivo ya existe no se vuelve a descargar.
# Para actualizar una fuente, borrar su archivo de data/raw/.
# ============================================================

ctx = ssl.create_default_context()
yr1 = urllib.request.urlopen("http://yr1.i.lencr.org/", timeout=60).read()
ctx.load_verify_locations(cadata=ssl.DER_cert_to_PEM_cert(yr1))

for nombre, url in FUENTES.items():
    destino = RAW / nombre
    if not destino.exists():
        tmp = destino.with_suffix(".part")
        with urllib.request.urlopen(url, context=ctx, timeout=600) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f)
        tmp.rename(destino)  # evita dejar archivos a medias si falla la descarga
    print(f"{nombre:28} {destino.stat().st_size / 1e6:8.1f} MB")

In [ ]:
# ============================================================
# 4. DESCOMPRESIÓN (INEGI)
# ============================================================

for zip_name in ["igae_mensual_csv.zip", "inpc_mensual_csv.zip"]:
    with zipfile.ZipFile(RAW / zip_name) as z:
        z.extractall(RAW / zip_name.removesuffix(".zip"))

RUTA_IGAE = next((RAW / "igae_mensual_csv" / "conjunto_de_datos").glob("conjunto_de_datos_igae_*.csv"))
RUTA_INPC = RAW / "inpc_mensual_csv" / "conjunto_de_datos" / "conjunto_de_datos_inpc_mensual.csv"
print(RUTA_IGAE.name)
print(RUTA_INPC.name)

In [ ]:
# ============================================================
# 5. SAT: XLS → CSV
# ============================================================
#
# Spark no lee Excel de forma nativa. El archivo tiene:
# - título en las filas 1-4;
# - encabezado en la fila 5 (y subencabezados de IEPS en la 6);
# - notas al pie al final.
#
# Solo cambiamos de formato y quitamos notas; la limpieza
# (meses en texto, cifras preliminares) va en transformaciones.
# ============================================================

sat_xls = pd.read_excel(RAW / "IngresosTributarios.xls", header=None).dropna(axis=1, how="all")

encabezado = sat_xls.iloc[5].fillna(sat_xls.iloc[4]).str.strip()
sat = sat_xls.iloc[6:].set_axis(encabezado, axis=1)
sat = sat[sat["Mes"].notna()]  # las notas al pie no tienen mes

RUTA_SAT = RAW / "ingresos_tributarios_sat.csv"
sat.to_csv(RUTA_SAT, index=False)

print(sat.shape)
sat.head()

In [ ]:
# ============================================================
# 6. SPARK SESSION
# ============================================================

spark = (
    SparkSession.builder
    .appName("Integrador_01_Ingesta")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark

In [ ]:
# ============================================================
# 7. LECTURA CON SPARK
# ============================================================
#
# inferSchema por ahora: es exploración. Cuando el pipeline
# esté estable conviene declarar el esquema explícito
# (ver docs/guia_optimizacion.md).
# ============================================================

leer = lambda ruta: spark.read.csv(str(ruta), header=True, inferSchema=True)

dfs = {
    "shcp": leer(RAW / "ingreso_gasto_finan.csv"),
    "sat": leer(RUTA_SAT),
    "igae": leer(RUTA_IGAE),
    "inpc": leer(RUTA_INPC),
}

for nombre, df in dfs.items():
    print(f"{nombre:5} filas={df.count():>9,}  columnas={len(df.columns)}")

In [ ]:
dfs["shcp"].printSchema()
dfs["inpc"].printSchema()

In [ ]:
# ============================================================
# 8. COBERTURA TEMPORAL
# ============================================================
#
# Cada fuente expresa el tiempo de forma distinta:
# SHCP → CICLO + MES (texto)   SAT → Año + Mes (texto)
# INPC → FECHA                 IGAE → columnas "AAAA|Mes" (formato ancho)
#
# Aquí solo verificamos que cubran el periodo de estudio
# (AAAAMM = año * 100 + mes).
# La llave común `fecha` se construye en transformaciones.
# ============================================================

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
         "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
mes_num = F.create_map(*[x for i, m in enumerate(MESES, 1) for x in (F.lit(m), F.lit(i))])

def rango(df, anio, mes):
    # (min, max) como AAAAMM. Falla si algún mes no se pudo interpretar.
    r = df.select((anio * 100 + mes).alias("p")).agg(F.min("p"), F.max("p"), F.count_if(F.col("p").isNull())).first()
    assert r[2] == 0, f"{r[2]} filas con año/mes no reconocido"
    return r[0], r[1]

sat_anio = F.regexp_extract(F.col("Año").cast("string"), r"(\d{4})", 1).cast("int")  # algunos años traen "*" (preliminar)
inpc = dfs["inpc"].where(F.col("CONCEPTO").endswith("Precios al Consumidor (INPC)"))

# IGAE: las columnas llegan a diciembre aunque estén vacías; tomamos los meses con dato
# de la serie total. Los encabezados traen marcas <R> (revisado) y <P> (preliminar).
igae_total = dfs["igae"].where(F.col("Descriptores") == "Índice de volumen físico base 2018=100|Total").first().asDict()
igae_p = [int(c[:4]) * 100 + MESES.index(c[5:].split("<")[0]) + 1
          for c, v in igae_total.items() if "|" in c and "Anual" not in c and v is not None]

rangos = {
    "shcp": rango(dfs["shcp"], F.col("CICLO"), mes_num[F.col("MES")]),
    "sat": rango(dfs["sat"], sat_anio, mes_num[F.col("Mes")]),
    "inpc": rango(inpc, F.year("FECHA"), F.month("FECHA")),
    "igae": (min(igae_p), max(igae_p)),
}

for nombre, (ini, fin) in rangos.items():
    ok = ini <= PERIODO[0] and fin >= PERIODO[1]
    print(f"{nombre:5} {ini} a {fin}  {'OK' if ok else 'NO CUBRE'}")
    assert ok, f"{nombre} no cubre el periodo {PERIODO}"


## Siguientes pasos

- `02_transformaciones`: llave `fecha` mensual en las cuatro fuentes, pivot de SHCP (XBC), homologar unidades a millones, IGAE de ancho a largo (serie total).
- Validaciones de calidad: nulos, duplicados por fecha y meses faltantes.
- Recortar todas las fuentes al periodo 2015-01 a 2025-03.